# 完整数据复现与交易核验

此Notebook需要未公开的完整历史输入、交易账本和补充实验明细。公开包请运行 `research.ipynb`。当前保存的输出已清除；先阅读RUN_GUIDE.md并准备已授权数据。

In [ ]:
from pathlib import Path
from microcap.preflight import require_full_inputs
require_full_inputs(Path.cwd(), 'evaluate')

# A股小市值研究：样本内选组与样本外验证
五档全部参与2015—2023样本内选优：单边30bp成本后月收益年化夏普最高者胜出，随后冻结市值区间及交易规则，用于2024—2025样本外（历史留出）验证；仍每月按当时资料重选股票，不是持续持有2023年的名单。其余四档样本外结果仅作描述性对照，不更换赢家。

2024—2025结果在研究过程中已查看，不宣称未窥视或完全独立验证。日线计划在前收盘形成，卖款从下一交易日使用。

In [ ]:
from pathlib import Path
import json, pandas as pd, numpy as np
from microcap.analysis import metrics, paired_tests
from microcap.portfolio import select_winner
from microcap.diagnostics import daily_risk, ablation_tests
root=Path.cwd()
c=json.loads((root/'config.json').read_text())
m=pd.read_csv(root/'results/monthly.csv',float_precision='round_trip')
s=json.loads((root/'results/selection.json').read_text())
assert set(m.strategy)==set(c['buckets'])|{'CSI1000'}
assert set(c['candidates'])==set(c['buckets'])==set(s['scores'])
assert len(c['candidates'])==5 and c['main_cost_bps']==30
assert m.return_month.nunique()==132
main=m[m.cost_bps.eq(30)]
s['winner']

## 仅用样本内独立复算五档评分
评分为√12×月净收益均值÷样本标准差，无风险率0。108个月完整样本；完全并列时，按低换手、较大市值区间排序。年化复合收益不参与选优。

In [ ]:
training=main[main.return_month.between(c['train_start'],c['train_end'])]
score_rows=[]
for strategy in c['candidates']:
    x=training[training.strategy.eq(strategy)]
    assert len(x)==108 and x.net_return.notna().all()
    score_rows.append(dict(strategy=strategy,score=np.sqrt(12)*x.net_return.mean()/x.net_return.std(ddof=1),turnover=x.turnover.mean(),n_train=len(x),rank_lower=c['buckets'][strategy][0]))
scores=pd.DataFrame(score_rows).sort_values(['score','turnover','rank_lower','strategy'],ascending=[False,True,False,True]).reset_index(drop=True)
assert scores.iloc[0].strategy==s['winner']
published=pd.read_csv(root/'results/selection_scores.csv').set_index('strategy')
for row in scores.itertuples():
    assert np.isclose(row.score,s['scores'][row.strategy]['score'])
    assert np.isclose(row.score,published.loc[row.strategy,'score'])
scores

In [ ]:
replay=select_winner(main,c)
assert replay['winner']==s['winner'] and replay['training_hash']==s['training_hash']
changed=main.copy()
test_mask=changed.return_month.between(c['test_start'],c['test_end'])
changed.loc[test_mask,'net_return']=.99
changed.loc[test_mask,'turnover']=999.
altered=select_winner(changed,c)
assert altered['winner']==s['winner']
assert altered['training_hash']==replay['training_hash']
print('修改全部样本外收益与换手后，五档样本内评分、训练输入哈希和赢家不变')

In [ ]:
r=pd.read_csv(root/'data/prepared/ranks.csv')
assert r.groupby('return_month').size().eq(500).all()
assert not r.duplicated(['return_month','stock_code']).any()
with np.load(root/'data/prepared/daily_market.npz') as z:
    di=pd.DatetimeIndex(z['dates']).get_indexer(pd.to_datetime(r.signal_date))
    si=r.stock_code.map({x:i for i,x in enumerate(z['codes'])}).to_numpy()
    assert (z['signal_state'][di,si]==0).all()
    assert not z['signal_known_ST'][di,si].any()
r.head()

In [ ]:
stats=metrics(m,c)
stats[stats.cost_bps.eq(30)&stats.period.isin(['TRAIN','TEST'])][['strategy','period','cagr','sharpe','max_drawdown']]

In [ ]:
for strategy,g in main.groupby('strategy'):
    x=g[g.return_month.between(c['test_start'],c['test_end'])].net_return
    expected=np.sqrt(12)*x.mean()/x.std(ddof=1)
    row=stats[stats.strategy.eq(strategy)&stats.period.eq('TEST')&stats.cost_bps.eq(30)].iloc[0]
    assert np.isclose(row.sharpe,expected)
print('Independent Sharpe checks passed; annual risk-free return=0')

## 冻结赢家的样本外验证
比较同一赢家的样本内与样本外表现；其余四档只作描述性对照。收益与夏普分别判断，不把收益排序直接解释成风险调整后优势。

In [ ]:
validation=pd.read_csv(root/'results/winner_validation.csv')
assert len(validation)==2 and validation.strategy.eq(s['winner']).all()
assert list(validation.period)==['TRAIN','TEST']
for row in validation.itertuples():
    expected=stats[stats.strategy.eq(s['winner'])&stats.period.eq(row.period)&stats.cost_bps.eq(30)].iloc[0]
    assert np.isclose(row.cagr,expected.cagr) and np.isclose(row.sharpe,expected.sharpe)
test_stats=stats[stats.strategy.isin(c['buckets'])&stats.period.eq('TEST')&stats.cost_bps.eq(30)].set_index('strategy')
print('赢家样本外收益排名：',int(test_stats.cagr.rank(ascending=False,method='min').loc[s['winner']]))
print('赢家样本外夏普排名：',int(test_stats.sharpe.rank(ascending=False,method='min').loc[s['winner']]))
validation[['strategy','period','cagr','sharpe','daily_max_drawdown','index_cagr','index_sharpe']]

In [ ]:
daily=pd.read_csv(root/'results/daily_nav.csv')
dr=daily_risk(daily,c)
dr[dr.period.eq('TEST')][['strategy','daily_max_drawdown','drawdown_peak','drawdown_trough','recovery_date','mean_cash','min_holdings','max_holdings']]

In [ ]:
tr=pd.read_csv(root/'results/trades.csv')
assert np.allclose(tr.cost,tr.notional*.003)
assert (pd.to_datetime(tr.plan_date)<pd.to_datetime(tr.date)).all()
assert (pd.to_datetime(tr.funding_asof)<pd.to_datetime(tr.date)).all()
rep=tr[tr.reason.eq('ST_replacement')]
for x in rep.itertuples():
    sales=tr[tr.strategy.eq(x.strategy)&tr.stock_code.eq(x.replaces)&tr.reason.eq('ST')&tr.date.lt(x.date)]
    assert len(sales)>0
rep.head()

In [ ]:
pairs=pd.DataFrame(paired_tests(m,s['winner'],c)['comparisons'])
pairs[pairs.period.eq('TEST')&pairs.comparator.eq('CSI1000')]

## 五档敏感性与ST规则对照
以下实验用于解释规则和假设的影响，不用于按样本外结果更换冻结赢家。

In [ ]:
pd.read_csv(root/'results/cost_sensitivity.csv').query("period=='TEST'").pivot(index='strategy',columns='cost_bps',values='cagr')

In [ ]:
pd.read_csv(root/'results/terminal_sensitivity.csv').query("period=='TEST' and strategy!='CSI1000'").pivot(index='strategy',columns='terminal_fraction',values='cagr')

In [ ]:
a=pd.read_csv(root/'results/st_ablation_monthly.csv')
base=a[a.rule.eq('EXIT_REPLACE')].sort_values(['strategy','return_month'])
expected=main[main.strategy.isin(c['buckets'])].sort_values(['strategy','return_month'])
assert np.allclose(base.net_return,expected.net_return)
pd.read_csv(root/'results/st_ablation_performance.csv').query("period=='TEST'")[[ 'strategy','rule','cagr','sharpe','daily_max_drawdown','mean_cash']]

In [ ]:
ablation_tests(a,c).query("period=='TEST'")

In [ ]:
coverage=pd.read_csv(root/'results/regression_coverage.csv')
coverage.head()

In [ ]:
pd.read_csv(root/'results/regression_summary.csv').query("period=='TEST'")

In [ ]:
json.loads((root/'results/quality.json').read_text())

解释边界：2024—2025可用于冻结赢家的样本外历史评价，但结果在研究过程中已查看，不能宣称未窥视或完全独立验证。价格指数与含分红复权策略口径不同；ST对照为观察性规则实验；无容量与盘中立即成交认证；剩余收益不证明壳价值。

## 扩展区间与资金执行敏感性
新增501—1000名只作描述性扩展，不进入原五档选优。资金实验只使用冻结赢家，两段分别从现金开始；整手与最低5元佣金不改变选股规则。佣金2.5bp包含在单边30bp中，额外费用仅为不足5元的差额。复权收益记账仍假设分配收益再投资，不是完整券商结算。

In [ ]:
import hashlib
sup=root/'results/supplement'
manifest=json.loads((sup/'manifest.json').read_text())
assert manifest['protocol']['frozen_winner']==s['winner']
for name,digest in manifest['sources'].items():
    assert hashlib.sha256((root/name).read_bytes()).hexdigest()==digest,name
for name,digest in manifest['outputs'].items():
    assert hashlib.sha256((root/name).read_bytes()).hexdigest()==digest,name
capital=pd.read_csv(sup/'capital_performance.csv')
cm=pd.read_csv(sup/'capital_monthly.csv')
for key,g in cm.groupby(['period','mode','initial_capital']):
    row=capital[(capital.period==key[0])&(capital['mode']==key[1])&(capital.initial_capital==key[2])].iloc[0]
    assert len(g)==(108 if key[0]=='TRAIN' else 24)
    assert np.isclose(row.cagr,(1+g.net_return).prod()**(12/len(g))-1)
    assert np.isclose(row.sharpe,np.sqrt(12)*g.net_return.mean()/g.net_return.std(ddof=1))
checks=pd.read_csv(sup/'capital_checks.csv')
assert checks.unverified_stale_marks.eq(0).all()
assert checks.buy_lot_max_error.dropna().lt(1e-7).all()
assert checks.min_cash_yuan.ge(-1e-5).all()
capital.query("period=='TEST'")[['mode','initial_capital','cagr','sharpe','daily_max_drawdown','mean_cash','mean_holdings']]

In [ ]:
extended=pd.read_csv(sup/'extended_monthly.csv')
assert len(extended)==5*132 and extended.net_return.notna().all()
assert not extended.duplicated(['strategy','return_month']).any()
assert not set(extended.strategy)&set(c['candidates'])
em=metrics(extended,c)
published=pd.read_csv(sup/'extended_performance.csv')
check=em.merge(published,on=['strategy','cost_bps','period'],suffixes=('_recalc','_saved'),validate='one_to_one')
assert np.allclose(check.cagr_recalc,check.cagr_saved)
assert np.allclose(check.sharpe_recalc,check.sharpe_saved)
published.query("period in ['TRAIN','TEST']")[['strategy','period','cagr','sharpe','daily_max_drawdown']]

In [ ]:
from IPython.display import Image, display
for name in ['extended_bands','capital_sensitivity']:
    display(Image(filename=str(root/'results/word_figures'/f'{name}.png')))
print('扩展只用于描述；资金模型不构成真实容量上限或独立样本外证明。')

## 策略适用性补充分析

固定原赢家和交易规则；以下为收益集中度、市场环境与执行约束的描述分析。运行 `python scripts/analyze_applicability.py` 可复算。完整口径见 RUN_GUIDE.md。当月标签仅作事后解释，不用于交易。

In [ ]:
from pathlib import Path
import pandas as pd
app_root = Path.cwd()
if not (app_root / 'results/applicability').exists():
    raise FileNotFoundError('请从项目根目录打开Notebook并先运行适用性分析脚本')
display(pd.read_csv(app_root / 'results/applicability/concentration.csv'))
display(pd.read_csv(app_root / 'results/applicability/annual_returns.csv'))

In [ ]:
environment = pd.read_csv(app_root / 'results/applicability/environment.csv')
display(environment[environment['period'].eq('TEST')])
display(pd.read_csv(app_root / 'results/applicability/execution_costs.csv'))
display(pd.read_csv(app_root / 'results/applicability/capital_constraints.csv').query("period == 'TEST'"))

## 成交容量与额外冲击：完整重跑和账本核验

固定原赢家，不能在此重新选组。先完成授权历史行情、公告证据与原回测输入校验，再运行独立入口：

```bash
python scripts/prepare_capacity_inputs.py --data-root /path/to/full/project
python scripts/run_capacity_stress.py --data-root /path/to/full/project --workers 2
```

新行情存于外部项目的`data/capacity/`，完整模拟账本存于发布代码工作副本的`.build/capacity_runs/`；原配置和主收益表保持不变。`run_capacity_stress.py`逐格调用`verify_capacity_ledgers`，重建每日现金、持仓与净值，检查前收预算、整手、共享成交上限和费用，然后才允许汇总发布。只检查最终收益相同，不能替代这些账目核验。

公开Notebook仅复算汇总指标；这里的完整重跑还需要未公开数据。日频VWAP和未校准的额外平方根冲击是压力假设，不能认证盘中真实成交。各期分别现金起步，8档本金随净值复利变化；完整定义见`results/capacity/protocol.json`。

In [ ]:
from scripts.verify_capacity import verify_capacity
capacity_root = Path.cwd()
display(verify_capacity(capacity_root))
capacity_performance = pd.read_csv(capacity_root/'results/capacity/performance.csv')
display(capacity_performance.query("period == 'TEST' and scenario == 'MAIN_1P'")[
    ['initial_capital','max_nav_yuan','cagr','sharpe','daily_max_drawdown',
     'mean_cash','attempt_completion','month_end_mean_target_shortfall_weight']])
display(pd.read_csv(capacity_root/'results/capacity/ledger_checks.csv'))
from scripts.verify_capacity_risk import verify_capacity_risk
display(verify_capacity_risk(capacity_root))
display(pd.read_csv(capacity_root/"results/capacity/terminal_cases.csv"))